In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATASET_PATH = Path(r"D:\FINAL DFDT\SELECTED_DATASET")

manifest = pd.read_csv(DATASET_PATH / "selection_manifest.csv")

manifest["class"] = manifest["class"].str.upper()
manifest["source_subfolder"] = manifest["source_subfolder"].astype(str)

print("Total images:", len(manifest))
display(manifest.head())

Total images: 10000


,class,source_subfolder,source_image,destination_image
0,REAL,000,E:\FINAL DATA1\real\000\191.png,E:\FINAL DATA1\SELECTED_DATASET\real\000_01_19...
1,REAL,000,E:\FINAL DATA1\real\000\178.png,E:\FINAL DATA1\SELECTED_DATASET\real\000_02_17...
2,REAL,000,E:\FINAL DATA1\real\000\101.png,E:\FINAL DATA1\SELECTED_DATASET\real\000_03_10...
3,REAL,000,E:\FINAL DATA1\real\000\076.png,E:\FINAL DATA1\SELECTED_DATASET\real\000_04_07...
4,REAL,000,E:\FINAL DATA1\real\000\063.png,E:\FINAL DATA1\SELECTED_DATASET\real\000_05_06...


In [2]:
# Group images by class and original source folder

group_sizes = (
    manifest.groupby(["class", "source_subfolder"])
    .size()
    .reset_index(name="image_count")
)

# Confirm the observed group-size distributions
print("Group-size distribution:")
display(
    group_sizes.groupby(["class", "image_count"])
    .size()
    .reset_index(name="number_of_groups")
)

# Exact allocation plan
allocation = {
    "REAL": {
        5: {"train": 694, "validation": 150, "test": 150},
        6: {"train": 5, "validation": 0, "test": 0}
    },
    "FAKE": {
        2: {"train": 0, "validation": 1, "test": 0},
        5: {"train": 700, "validation": 146, "test": 144},
        6: {"train": 0, "validation": 3, "test": 5}
    }
}

rng = np.random.default_rng(42)

group_assignments = []

for class_name, size_plan in allocation.items():

    for image_count, split_plan in size_plan.items():

        matching_groups = group_sizes[
            (group_sizes["class"] == class_name) &
            (group_sizes["image_count"] == image_count)
        ].copy()

        matching_groups = matching_groups.sample(
            frac=1,
            random_state=42
        ).reset_index(drop=True)

        position = 0

        for split_name, number_of_groups in split_plan.items():

            selected = matching_groups.iloc[
                position:position + number_of_groups
            ]

            for _, group in selected.iterrows():
                group_assignments.append({
                    "class": class_name,
                    "source_subfolder": group["source_subfolder"],
                    "image_count": image_count,
                    "split": split_name
                })

            position += number_of_groups

        if position != len(matching_groups):
            raise ValueError(
                f"Allocation mismatch: {class_name}, "
                f"group size {image_count}"
            )

group_split_df = pd.DataFrame(group_assignments)

print("Total assigned source groups:", len(group_split_df))
display(group_split_df.groupby(["class", "split"]).size().unstack())

Group-size distribution:


,class,image_count,number_of_groups
0,FAKE,2,1
1,FAKE,5,990
2,FAKE,6,8
3,REAL,5,994
4,REAL,6,5


Total assigned source groups: 1998


split,test,train,validation
class,,,
FAKE,149,700,150
REAL,150,699,150


In [3]:
split_manifest = manifest.merge(
    group_split_df[["class", "source_subfolder", "split"]],
    on=["class", "source_subfolder"],
    how="left",
    validate="many_to_one"
)

if split_manifest["split"].isna().any():
    raise ValueError("Some images have no split assignment!")

print("Total assigned images:", len(split_manifest))

display(
    split_manifest.groupby(["split", "class"])
    .size()
    .unstack(fill_value=0)
)

Total assigned images: 10000


class,FAKE,REAL
split,,
test,750,750
train,3500,3500
validation,750,750


In [4]:
expected = {
    ("train", "REAL"): 3500,
    ("train", "FAKE"): 3500,
    ("validation", "REAL"): 750,
    ("validation", "FAKE"): 750,
    ("test", "REAL"): 750,
    ("test", "FAKE"): 750
}

actual = split_manifest.groupby(
    ["split", "class"]
).size().to_dict()

for key, expected_count in expected.items():
    actual_count = actual.get(key, 0)

    print(
        f"{key[0]:10s} | {key[1]:5s} | "
        f"Expected: {expected_count} | Actual: {actual_count}"
    )

    assert actual_count == expected_count, (
        f"Count mismatch for {key}"
    )

print("\nSUCCESS: All dataset split counts are exact!")

train      | REAL  | Expected: 3500 | Actual: 3500
train      | FAKE  | Expected: 3500 | Actual: 3500
validation | REAL  | Expected: 750 | Actual: 750
validation | FAKE  | Expected: 750 | Actual: 750
test       | REAL  | Expected: 750 | Actual: 750
test       | FAKE  | Expected: 750 | Actual: 750

SUCCESS: All dataset split counts are exact!


In [5]:
train_groups = set(
    map(tuple, split_manifest.loc[
        split_manifest["split"] == "train",
        ["class", "source_subfolder"]
    ].values)
)

val_groups = set(
    map(tuple, split_manifest.loc[
        split_manifest["split"] == "validation",
        ["class", "source_subfolder"]
    ].values)
)

test_groups = set(
    map(tuple, split_manifest.loc[
        split_manifest["split"] == "test",
        ["class", "source_subfolder"]
    ].values)
)

print("Train-validation overlap:", len(train_groups & val_groups))
print("Train-test overlap:", len(train_groups & test_groups))
print("Validation-test overlap:", len(val_groups & test_groups))

assert len(train_groups & val_groups) == 0
assert len(train_groups & test_groups) == 0
assert len(val_groups & test_groups) == 0

print("\nSUCCESS: No source-group leakage!")

Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0

SUCCESS: No source-group leakage!


In [6]:
OUTPUT_PATH = DATASET_PATH / "splits"
OUTPUT_PATH.mkdir(exist_ok=True)

split_manifest.to_csv(
    OUTPUT_PATH / "dataset_split_manifest.csv",
    index=False
)

group_split_df.to_csv(
    OUTPUT_PATH / "source_group_assignments.csv",
    index=False
)

print("Split manifests saved successfully.")
print(OUTPUT_PATH)

Split manifests saved successfully.
D:\FINAL DFDT\SELECTED_DATASET\splits


In [7]:
OUTPUT_PATH = DATASET_PATH / "splits"
OUTPUT_PATH.mkdir(exist_ok=True)

split_manifest.to_csv(
    OUTPUT_PATH / "dataset_split_manifest.csv",
    index=False
)

group_split_df.to_csv(
    OUTPUT_PATH / "source_group_assignments.csv",
    index=False
)

print("Saved successfully!")
print(OUTPUT_PATH)

Saved successfully!
D:\FINAL DFDT\SELECTED_DATASET\splits
